# CATE Exploration with Causal Forests: PISA 2022 Germany

Treatment: `PRESCHOOL`  
Outcome: `READ`

In [ ]:
# Install econml if needed
# !pip install econml

# Import packages

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance

RANDOM_STATE = 123

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)

## 1. Load data and define variables

In [ ]:
# Load data

file = "PISA2022deu.csv"
data_raw = pd.read_csv(file)

print("Raw data:")
print(f"Rows:    {data_raw.shape[0]}")
print(f"Columns: {data_raw.shape[1]}")

# Recode treatment

data_raw["PREEDU"] = pd.to_numeric(data_raw["PREEDU"], errors="coerce")
data_raw["PRESCHOOL"] = 1 - data_raw["PREEDU"]

display(data_raw.head())

In [ ]:
# Define variables

treatment = "PRESCHOOL"
outcome = "READ"

covariates = [
    "GENDER",
    "AGE",
    "IMMIG",
    "HISCED",
    "HOMEPOS",
    "ESCS",
    "CREATHME",
    "PA188Q03JA",
    "PA005Q01TA",
    "HISEI",
]

analysis_vars = [treatment, outcome] + covariates

missing_columns = [v for v in analysis_vars if v not in data_raw.columns]
if missing_columns:
    raise ValueError(f"Missing variables in the data: {missing_columns}")

data = data_raw[analysis_vars + ["PREEDU"]].copy()

display(pd.DataFrame({
    "Variable": analysis_vars,
    "Role": ["Treatment", "Outcome"] + ["Covariate"] * len(covariates),
    "Dtype": [str(data[v].dtype) for v in analysis_vars],
    "Unique values": [data[v].nunique(dropna=True) for v in analysis_vars],
}))

## 2. Complete-case sample

In [ ]:
# Missing-value table

missing_table = pd.DataFrame({
    "Missing N": data[analysis_vars].isna().sum(),
    "Missing %": data[analysis_vars].isna().mean() * 100,
    "Non-missing N": data[analysis_vars].notna().sum()
}).sort_values("Missing %", ascending=False)

display(missing_table.round(2))

In [ ]:
# Build complete-case sample

data_cc = data.dropna(subset=analysis_vars).copy()

data_cc[treatment] = pd.to_numeric(data_cc[treatment], errors="coerce").astype(int)
data_cc[outcome] = pd.to_numeric(data_cc[outcome], errors="coerce")

data_cc = data_cc[data_cc[treatment].isin([0, 1])].copy()

print("Complete-case sample:")
print(f"Rows before: {data.shape[0]}")
print(f"Rows after:  {data_cc.shape[0]}")
print(f"Dropped:     {data.shape[0] - data_cc.shape[0]} ({(1 - data_cc.shape[0] / data.shape[0]) * 100:.2f}%)")

pre_table = pd.DataFrame({
    "N": data_cc[treatment].value_counts().sort_index(),
    "%": data_cc[treatment].value_counts(normalize=True).sort_index() * 100
})
pre_table.index = ["No preschool (0)", "Preschool (1)"][:len(pre_table)]

display(pre_table.round(2))

## 3. Prepare covariates

In [ ]:
# Define covariate types

categorical_covariates = [
    "GENDER",
    "IMMIG",
    "HISCED",
    "PA188Q03JA",
    "PA005Q01TA",
]

numeric_covariates = [
    "AGE",
    "HOMEPOS",
    "ESCS",
    "CREATHME",
    "HISEI",
]

categorical_covariates = [v for v in categorical_covariates if v in covariates]
numeric_covariates = [v for v in numeric_covariates if v in covariates]

# Build feature matrix

X_raw = data_cc[covariates].copy()

for v in numeric_covariates:
    X_raw[v] = pd.to_numeric(X_raw[v], errors="coerce")

for v in categorical_covariates:
    X_raw[v] = X_raw[v].astype("category")

X = pd.get_dummies(X_raw, columns=categorical_covariates, drop_first=True)
X = X.astype(float)

Y = data_cc[outcome].astype(float).values
T = data_cc[treatment].astype(int).values

print("Feature matrix:")
print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")
print(f"T shape: {T.shape}")

display(X.head())

In [ ]:
# Split data

X_train, X_test, Y_train, Y_test, T_train, T_test, idx_train, idx_test = train_test_split(
    X,
    Y,
    T,
    data_cc.index,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=T
)

print("Training sample:")
print(f"N = {len(Y_train)}")
print(pd.Series(T_train).value_counts().sort_index().rename({0: "No preschool", 1: "Preschool"}))

print("\nTest sample:")
print(f"N = {len(Y_test)}")
print(pd.Series(T_test).value_counts().sort_index().rename({0: "No preschool", 1: "Preschool"}))

## 4. Descriptive benchmark

In [ ]:
# Compute descriptive mean difference

test_tmp = pd.DataFrame({
    "READ": Y_test,
    "PRESCHOOL": T_test
})

desc_test = test_tmp.groupby("PRESCHOOL")["READ"].agg(["count", "mean", "std"])
desc_test.index = ["No preschool (0)", "Preschool (1)"][:len(desc_test)]

display(desc_test.round(2))

if set(np.unique(T_test)) == {0, 1}:
    diff = test_tmp.loc[test_tmp["PRESCHOOL"] == 1, "READ"].mean() - test_tmp.loc[test_tmp["PRESCHOOL"] == 0, "READ"].mean()
    print(f"Descriptive mean difference, preschool minus no preschool: {diff:.2f} READ points")

## 5. Causal Forest DML

In [ ]:
# Import econml

try:
    from econml.dml import CausalForestDML
    econml_available = True
    print("econml is available.")
except ImportError:
    econml_available = False
    print("econml is not installed.")
    print("Install it with: !pip install econml")

In [ ]:
# Estimate CausalForestDML

if econml_available:
    model_y = RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model_t = RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    cf = CausalForestDML(
        model_y=model_y,
        model_t=model_t,
        discrete_treatment=True,
        n_estimators=800,
        min_samples_leaf=10,
        max_depth=None,
        cv=3,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    cf.fit(Y_train, T_train, X=X_train)

    cate_cf = np.ravel(cf.effect(X_test))
    ate_cf_test = np.mean(cate_cf)

    print(f"Average predicted CATE in the test sample: {ate_cf_test:.2f} READ points")

    try:
        lb_cf, ub_cf = cf.effect_interval(X_test, alpha=0.05)
        lb_cf = np.ravel(lb_cf)
        ub_cf = np.ravel(ub_cf)
        print(f"Mean 95% interval across test observations: [{np.mean(lb_cf):.2f}, {np.mean(ub_cf):.2f}]")
    except Exception:
        lb_cf, ub_cf = None, None
        print("CATE intervals were not available.")
else:
    cate_cf = None
    lb_cf, ub_cf = None, None

## 6. X-Learner with random forests

In [ ]:
# Estimate X-Learner

if econml_available:
    from econml.metalearners import XLearner

    x_learner = XLearner(
        models=RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=10,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        cate_models=RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=10,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        propensity_model=RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=10,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    x_learner.fit(Y_train, T_train, X=X_train)

    cate_x = np.ravel(x_learner.effect(X_test))
    ate_x_test = np.mean(cate_x)

    print(f"Average predicted X-Learner CATE in the test sample: {ate_x_test:.2f} READ points")
else:
    cate_x = None

## 7. CATE results

In [ ]:
# Combine CATE estimates with test-sample data

cate_df = data_cc.loc[idx_test, analysis_vars].copy()
cate_df["PRESCHOOL_label"] = cate_df[treatment].map({0: "No preschool", 1: "Preschool"})

if cate_cf is not None:
    cate_df["CATE_CausalForestDML"] = cate_cf
    if lb_cf is not None and ub_cf is not None:
        cate_df["CATE_CF_lb95"] = lb_cf
        cate_df["CATE_CF_ub95"] = ub_cf

if cate_x is not None:
    cate_df["CATE_XLearner_RF"] = cate_x

display(cate_df.head())

In [ ]:
# Summarize CATE distributions

cate_cols = [c for c in ["CATE_CausalForestDML", "CATE_XLearner_RF"] if c in cate_df.columns]

if len(cate_cols) > 0:
    cate_summary = cate_df[cate_cols].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]).T
    display(cate_summary.round(2))
else:
    print("No CATE estimates available.")

In [ ]:
# Plot CATE distributions

for c in cate_cols:
    plt.figure(figsize=(7, 4))
    plt.hist(cate_df[c].dropna(), bins=30)
    plt.axvline(cate_df[c].mean(), linestyle="--", label="Mean")
    plt.axvline(0, linestyle=":", label="Zero effect")
    plt.xlabel("Estimated CATE in READ points")
    plt.ylabel("Frequency")
    plt.title(f"Distribution of estimated CATEs: {c}")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Compare CATE estimators

if {"CATE_CausalForestDML", "CATE_XLearner_RF"}.issubset(cate_df.columns):
    corr_cates = cate_df[["CATE_CausalForestDML", "CATE_XLearner_RF"]].corr().iloc[0, 1]
    print(f"Correlation of CATE estimates: {corr_cates:.3f}")

    plt.figure(figsize=(6, 5))
    plt.scatter(cate_df["CATE_CausalForestDML"], cate_df["CATE_XLearner_RF"], alpha=0.5)
    plt.axhline(0, linestyle=":")
    plt.axvline(0, linestyle=":")
    plt.xlabel("CATE CausalForestDML")
    plt.ylabel("CATE X-Learner RF")
    plt.title("Comparison of CATE estimates")
    plt.tight_layout()
    plt.show()

## 8. CATE quintiles

In [ ]:
# Create CATE quintiles

main_cate = "CATE_CausalForestDML" if "CATE_CausalForestDML" in cate_df.columns else (
    "CATE_XLearner_RF" if "CATE_XLearner_RF" in cate_df.columns else None
)

if main_cate is not None:
    cate_df["CATE_quintile"] = pd.qcut(cate_df[main_cate], q=5, labels=["Q1 low", "Q2", "Q3", "Q4", "Q5 high"], duplicates="drop")

    quintile_summary = cate_df.groupby("CATE_quintile").agg(
        N=(main_cate, "count"),
        CATE_mean=(main_cate, "mean"),
        CATE_min=(main_cate, "min"),
        CATE_max=(main_cate, "max"),
        READ_mean=(outcome, "mean"),
        PRESCHOOL_share=(treatment, "mean")
    )

    display(quintile_summary.round(2))
else:
    print("No main CATE variable available.")

In [ ]:
# Profile CATE quintiles

if main_cate is not None:
    numeric_for_profile = ["AGE", "HOMEPOS", "ESCS", "CREATHME", "HISEI"]
    numeric_for_profile = [v for v in numeric_for_profile if v in cate_df.columns]

    profile_df = cate_df.copy()
    for v in numeric_for_profile:
        profile_df[v] = pd.to_numeric(profile_df[v], errors="coerce")

    profile_table = profile_df.groupby("CATE_quintile")[numeric_for_profile].mean()
    display(profile_table.round(2))

In [ ]:
# Plot average CATE by quintile

if main_cate is not None:
    plot_data = cate_df.groupby("CATE_quintile")[main_cate].mean()

    plt.figure(figsize=(7, 4))
    plt.bar(plot_data.index.astype(str), plot_data.values)
    plt.axhline(0, linestyle=":")
    plt.ylabel("Average estimated CATE")
    plt.title("Average CATE by quintile")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## 9. CATE by covariates

In [ ]:
# CATE by numeric covariates

if main_cate is not None:
    numeric_heterogeneity_vars = ["ESCS", "HOMEPOS", "HISEI", "AGE", "CREATHME"]
    numeric_heterogeneity_vars = [v for v in numeric_heterogeneity_vars if v in cate_df.columns]

    for v in numeric_heterogeneity_vars:
        tmp = cate_df[[v, main_cate]].copy()
        tmp[v] = pd.to_numeric(tmp[v], errors="coerce")
        tmp = tmp.dropna()

        if tmp[v].nunique() >= 5:
            try:
                tmp[f"{v}_quartile"] = pd.qcut(tmp[v], q=4, duplicates="drop")
                by_bin = tmp.groupby(f"{v}_quartile")[main_cate].agg(["count", "mean", "std"])
                display(by_bin.round(2))

                plt.figure(figsize=(7, 4))
                plt.bar(by_bin.index.astype(str), by_bin["mean"])
                plt.axhline(0, linestyle=":")
                plt.ylabel("Average estimated CATE")
                plt.title(f"CATE by quartiles of {v}")
                plt.xticks(rotation=30, ha="right")
                plt.tight_layout()
                plt.show()
            except Exception:
                print(f"{v}: quartiles could not be created.")
        else:
            print(f"{v}: not enough unique values for quartiles.")

In [ ]:
# CATE by categorical covariates

if main_cate is not None:
    categorical_heterogeneity_vars = ["GENDER", "IMMIG", "HISCED", "PA188Q03JA", "PA005Q01TA"]
    categorical_heterogeneity_vars = [v for v in categorical_heterogeneity_vars if v in cate_df.columns]

    for v in categorical_heterogeneity_vars:
        if cate_df[v].nunique(dropna=True) <= 12:
            by_group = cate_df.groupby(v)[main_cate].agg(["count", "mean", "std"]).sort_index()
            display(by_group.round(2))

            plt.figure(figsize=(7, 4))
            plt.bar(by_group.index.astype(str), by_group["mean"])
            plt.axhline(0, linestyle=":")
            plt.ylabel("Average estimated CATE")
            plt.title(f"CATE by {v}")
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            plt.show()
        else:
            print(f"{v}: too many categories for a compact plot.")

## 10. Feature importance

In [ ]:
# Causal-forest feature importances

if econml_available and "cf" in globals():
    try:
        importances = np.ravel(cf.feature_importances_)
        importance_df = pd.DataFrame({
            "Feature": X.columns,
            "Importance": importances
        }).sort_values("Importance", ascending=False)

        display(importance_df.head(20).round(4))

        top_imp = importance_df.head(15).sort_values("Importance")

        plt.figure(figsize=(8, 6))
        plt.barh(top_imp["Feature"], top_imp["Importance"])
        plt.xlabel("Feature importance")
        plt.title("Main features for treatment-effect heterogeneity")
        plt.tight_layout()
        plt.show()
    except Exception:
        print("Causal-forest feature importances were not available.")
else:
    print("Causal forest was not estimated.")

In [ ]:
# Permutation importance for CATE predictions

if main_cate is not None:
    cate_target = cate_df[main_cate].values
    X_test_for_imp = X.loc[cate_df.index]

    rf_cate_explainer = RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    rf_cate_explainer.fit(X_test_for_imp, cate_target)

    perm = permutation_importance(
        rf_cate_explainer,
        X_test_for_imp,
        cate_target,
        n_repeats=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    perm_df = pd.DataFrame({
        "Feature": X_test_for_imp.columns,
        "Permutation Importance Mean": perm.importances_mean,
        "Permutation Importance SD": perm.importances_std
    }).sort_values("Permutation Importance Mean", ascending=False)

    display(perm_df.head(20).round(4))

    top_perm = perm_df.head(15).sort_values("Permutation Importance Mean")

    plt.figure(figsize=(8, 6))
    plt.barh(top_perm["Feature"], top_perm["Permutation Importance Mean"])
    plt.xlabel("Permutation importance")
    plt.title("Features explaining CATE predictions")
    plt.tight_layout()
    plt.show()

## 11. Output tables

In [ ]:
# Export results

if main_cate is not None:
    cate_df.to_csv("pisa_cate_results_preschool_read.csv", index=False)
    print("Saved: pisa_cate_results_preschool_read.csv")

    if "quintile_summary" in globals():
        quintile_summary.to_csv("pisa_cate_quintile_summary.csv")
        print("Saved: pisa_cate_quintile_summary.csv")

    if "profile_table" in globals():
        profile_table.to_csv("pisa_cate_quintile_profiles.csv")
        print("Saved: pisa_cate_quintile_profiles.csv")

    if "importance_df" in globals():
        importance_df.to_csv("pisa_causal_forest_feature_importance.csv", index=False)
        print("Saved: pisa_causal_forest_feature_importance.csv")

    if "perm_df" in globals():
        perm_df.to_csv("pisa_cate_permutation_importance.csv", index=False)
        print("Saved: pisa_cate_permutation_importance.csv")
else:
    print("No export because no CATE estimates are available.")